In [27]:
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import pandas as pd
import json
import csv
import pickle
import numpy as np

counter = 0
core_name = "dc_cubes_historic"
predictionColumn = "cpuusage_ps"


In [42]:
def pushData(row):
    global core_name
    global counter
    global allMetrics
    global predictionColumn
    # defining the api-endpoint
    url = "http://localhost:8983/solr/"+core_name+"/update/json/docs"
    # data to be sent to api
    data = {
                "timestamp": str(row["timestamp"]),
                "cluster": row["cluster"],
                "dc": row["dc"],
                "perm": -1, #row["perm"],
                "instanz": row["instanz"],
                "verfahren": "-1",# row["verfahren"],
                "service": "-1" ,#row["verfahren"],
                "response": -1 ,#row["response"],
                "count": row[predictionColumn],
                "minv":  -1,#row["minv"],
                "maxv":  -1,#row["maxv"],
                "avg": -1, #row["avg"],
                "var": -1 ,#row["var"],
                "dev_upp": -1, #row["dev_upp"],
                "dev_low": -1, #row["dev_low"],
                "perc90": -1 ,#row["perc90"],
                "perc95": -1 ,#row["perc95"],
                "perc99": -1 ,#row["perc99.9"],
                "sum": -1, #row["sum"],
                "sum_of_squares": -1, #row["sum_of_squares"],
                "server": "-1", #row["server"]
            }
    
    for metric in allMetrics:
        data[metric] = row[metric]
    session = requests.Session()
    retry = Retry(connect=3, backoff_factor=0.5)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.get(url)
    headers = {'Content-type': 'application/json'}
    # sending post request
    session.post(url=url, data=json.dumps(data), headers=headers)
    counter += 1
    if (counter % 1000 == 0):
        print("Commiting... counter:", counter)
        requests.get("http://localhost:8983/solr/"+core_name+"/update?commit=true")

In [29]:
def createSolrCore(core_name):
    url = "http://localhost:8983/solr/admin/cores?action=CREATE&name=" + \
        core_name+"&configSet=_default"
    requests.post(url=url)
    print(core_name, " created")



In [30]:
def initSchema(core_name, allMetrics):
    url = "http://localhost:8983/solr/"+core_name+"/schema"
    headers = {'Content-type': 'application/json'}
    rowsDict = {
        "timestamp": "pdate", "host": "string", "cluster": "pint", "dc": "pint", "perm": "pint", "instanz": "string", "verfahren": "string",
        "service": "string", "response": "pint", "count": "pint", "minv": "pint", "maxv": "pint", "avg": "pfloat", "var": "pfloat",
        "dev_upp": "pfloat", "dev_low": "pfloat", "perc90": "pfloat", "perc95": "pfloat", "perc99": "pfloat", "sum": "pint",
        "sum_of_squares": "pint", "server": "string"}

    for name in rowsDict:
        data = {
            "add-field": {"stored": "true", "docValues": "true", "indexed": "false", "multiValued": "false", "name": name, "type": rowsDict[name]}
        }
        requests.post(url=url, data=json.dumps(data), headers=headers)
        
    for metric in allMetrics:
        data = {
            "add-field": {"stored": "true", "docValues": "true", "indexed": "false", "multiValued": "false", "name": metric, "type": "pfloat"}
        }
        requests.post(url=url, data=json.dumps(data), headers=headers)
    
    
    print(core_name, " schema inited")


In [61]:
def deleteCoreDocuments(core_name):
    url = "http://localhost:8983/solr/"+core_name + \
        "/update?commitWithin=1000&overwrite=true&wt=json"
    headers = {'Content-type': 'application/json'}
    data = {'delete': {'query': '*:*'}}
    requests.post(url=url, data=json.dumps(data), headers=headers)
    print("deleted old documents from "+core_name+" core")

In [6]:
# get existing solr cores
url = "http://localhost:8983/solr/admin/cores?action=STATUS"
response = requests.get(url).json()
activeCores = response['status'].keys()

In [7]:
if core_name in activeCores:
    print(core_name + " already exists")
    # delete old data/predictions
#     deleteCoreDocuments(core_name)
# else forecast core doesn't exist
else:
    print(core_name + " doesn't exist")
    # create an new forecast solr core
    createSolrCore(core_name)
    # init schema
    initSchema(core_name)

dc_cubes_historic doesn't exist
dc_cubes_historic  created
dc_cubes_historic  schema inited


In [8]:
# Read data from pickle file
with open("./4week_transformed_droppedErrors_filled.pkl", "rb") as pickleFile:
    df = pickle.load(pickleFile)
df = df.reset_index()

In [12]:
allMetrics = df.columns

In [15]:
allMetrics = allMetrics.drop("timestamp")

In [18]:
initSchema(core_name, allMetrics)

dc_cubes_historic  schema inited


In [66]:
# There are two dc's: 0 and 1
# dc 0 has clusters 6 and 8
# dc 1 has clusters 5 and 7
# Every Cluster has all 8 instances

def pushDataForAllInstances():
    instances = ["1","2","3","4","5","6","7","8" ]
    for instanz in instances:
        df["instanz"] = instanz
        df.apply(pushData, axis=1)

In [67]:
df["dc"] = 0

In [68]:
df["cluster"] = 6

In [69]:
pushDataForAllInstances()

Commiting... counter: 3000
Commiting... counter: 4000
Commiting... counter: 5000
Commiting... counter: 6000
Commiting... counter: 7000
Commiting... counter: 8000
Commiting... counter: 9000
Commiting... counter: 10000
Commiting... counter: 11000
Commiting... counter: 12000
Commiting... counter: 13000
Commiting... counter: 14000
Commiting... counter: 15000
Commiting... counter: 16000
Commiting... counter: 17000
Commiting... counter: 18000
Commiting... counter: 19000
Commiting... counter: 20000
Commiting... counter: 21000
Commiting... counter: 22000
Commiting... counter: 23000
Commiting... counter: 24000
Commiting... counter: 25000


In [70]:
df["cluster"] = 8

In [71]:
pushDataForAllInstances()

Commiting... counter: 26000
Commiting... counter: 27000
Commiting... counter: 28000
Commiting... counter: 29000
Commiting... counter: 30000
Commiting... counter: 31000
Commiting... counter: 32000
Commiting... counter: 33000
Commiting... counter: 34000
Commiting... counter: 35000
Commiting... counter: 36000
Commiting... counter: 37000
Commiting... counter: 38000
Commiting... counter: 39000
Commiting... counter: 40000
Commiting... counter: 41000
Commiting... counter: 42000
Commiting... counter: 43000
Commiting... counter: 44000
Commiting... counter: 45000
Commiting... counter: 46000
Commiting... counter: 47000


In [72]:
df["dc"] = 1

In [73]:
df["cluster"] = 5

In [74]:
pushDataForAllInstances()

Commiting... counter: 48000
Commiting... counter: 49000
Commiting... counter: 50000
Commiting... counter: 51000
Commiting... counter: 52000
Commiting... counter: 53000
Commiting... counter: 54000
Commiting... counter: 55000
Commiting... counter: 56000
Commiting... counter: 57000
Commiting... counter: 58000
Commiting... counter: 59000
Commiting... counter: 60000
Commiting... counter: 61000
Commiting... counter: 62000
Commiting... counter: 63000
Commiting... counter: 64000
Commiting... counter: 65000
Commiting... counter: 66000
Commiting... counter: 67000
Commiting... counter: 68000
Commiting... counter: 69000


In [75]:
df["cluster"] = 7

In [76]:
pushDataForAllInstances()

Commiting... counter: 70000
Commiting... counter: 71000
Commiting... counter: 72000
Commiting... counter: 73000
Commiting... counter: 74000
Commiting... counter: 75000
Commiting... counter: 76000
Commiting... counter: 77000
Commiting... counter: 78000
Commiting... counter: 79000
Commiting... counter: 80000
Commiting... counter: 81000
Commiting... counter: 82000
Commiting... counter: 83000
Commiting... counter: 84000
Commiting... counter: 85000
Commiting... counter: 86000
Commiting... counter: 87000
Commiting... counter: 88000
Commiting... counter: 89000
Commiting... counter: 90000
Commiting... counter: 91000
Commiting... counter: 92000
